In [1]:
from pathlib import Path
import sys
import time

from sklearn.metrics import roc_auc_score

PROJECT_ROOT = Path(
    "/Users/hongmo/Desktop/Gradient Boosting"
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_and_split_data
from src.preprocessing import build_preprocessor
from src.models import build_catboost_pipeline

In [2]:
data_path = PROJECT_ROOT / "data" / "application_train.csv"

X_train, X_val, X_test, y_train, y_val, y_test = (
    load_and_split_data(data_path)
)

# Reuse preprocessing pipeline
cat_preprocessor = build_preprocessor(
    X_train,
    scale_numeric=False
)

# Build CatBoost pipeline
cat_pipeline = build_catboost_pipeline(
    cat_preprocessor
)

In [3]:
start_time = time.perf_counter()
cat_pipeline.fit(X_train, y_train)
training_time = time.perf_counter() - start_time

print(f"CatBoost Training Time: {training_time:.2f} seconds")

CatBoost Training Time: 13.93 seconds


In [4]:
# Predict probabilities
y_train_prob = cat_pipeline.predict_proba(X_train)[:, 1]
y_val_prob = cat_pipeline.predict_proba(X_val)[:, 1]

# Calculate ROC-AUC
train_auc = roc_auc_score(y_train, y_train_prob)
val_auc = roc_auc_score(y_val, y_val_prob)

# Display results
print(f"Training AUC:   {train_auc:.4f}")
print(f"Validation AUC: {val_auc:.4f}")
print(f"AUC Gap:        {train_auc - val_auc:.4f}")
print(f"Training Time:  {training_time:.2f} seconds")

Training AUC:   0.7704
Validation AUC: 0.7561
AUC Gap:        0.0142
Training Time:  13.93 seconds
